In [ ]:
// load the formatting setup and the style sheet
#load "../setup.fsx"
HTML(System.IO.File.ReadAllText "../style.css")

# A Logic Puzzle

The following exercise is taken from the book 
<a href="https://www.amazon.de/Logeleien-Zweistein-ihren-Antworten-Wegner/dp/B006YF0VUE">"99 Logeleien von Zweistein"</a>.
This book has been published 1968.  It is written by 
<a href="http://de.wikipedia.org/wiki/Thomas_von_Randow">Thomas von Randow</a>.

---
The gentlemen Amann, Bemann, Cemann and Demann are called - not necessarily in the same order - by their first names Erich, Fritz, Gustav and Heiner. They are all married to exactly one woman. We also know the following about them and their wives:

- Either Amann's first name is Heiner, or Bemann's wife is Inge.
- If Cemann is married to Josefa, then - **and only in this case** - Klara's husband is **not** called Fritz.
- If Josefa's husband is **not** called Erich, then Inge is married to Fritz.
- If Luise's husband is called Fritz, then Klara's husband's first name is **not** Gustav.
- If the wife of Fritz is called Inge, then Erich is **not** married to Josefa.
- If Fritz is **not** married to Luise, then Gustav's wife's name is Klara.
- Either Demann is married to Luise, or Cemann is called Gustav.

*What are the full names of these gentlemen, and what are their wives' first names?*

---

We are going to solve this problem by coding it in propositional logic and we will solve the resulting set of clauses using the Davis-Putnam algorithm.  In order to code the problem, we will use the following propositional variables:

- $\texttt{MaleName<}x\texttt{,}z\texttt{>}$ for any male first name $x$ and any surname $z$ expresses
  that the gentleman with first name $x$ has surname $z$.
- $\texttt{Married<}x\texttt{,}y\texttt{>}$ for any male first name $x$ and any female first name $y$ expresses
  that the gentleman with first name $x$ is married to the lady with first name $y$.
- $\texttt{FemaleName<}y\texttt{,}z\texttt{>}$ for any female first name $y$ and any last name $z$ expresses
  that the lady with first name $y$ has surname $z$.

We are using the symbols $\texttt{<}$ and $\texttt{>}$ as part of the propositional variables because we want to show the structure of these variables and the parser for propositional logic accepts these symbols as part of propositional variables.

In [ ]:
let FirstMale   = set [ "Erich"; "Fritz"; "Gustav"; "Heiner" ]
let FirstFemale = set [ "Inge"; "Josefa"; "Klara"; "Luise" ]
let SurNames    = set [ "Amann"; "Bemann"; "Cemann"; "Demann" ]

We import the Davis-Putnam algorithm.  This also imports the notebooks `04-CNF.ipynb` and `Propositional-Logic-Parser.ipynb`.

In [ ]:
importNotebook "07-Davis-Putnam-JW.ipynb"

The function `makeVar` takes three arguments:
- `f` is the name of a relation, e.g. `"Married"`,
- `x` is a name,
- `y` is a name.

It returns the propositional variable `f<x,y>`.

In [ ]:
let makeVar (f: string) (x: string) (y: string) : string =
    f + "<" + x + "," + y + ">"

In [ ]:
makeVar "Married" "Heiner" "Klara"

The function `atMostOne` takes one argument:
- `S` is a set of propositional variables.

It returns a set of clauses expressing the fact that at most one of the variables of $S$ is `true`.

In [ ]:
let atMostOne (S: Set<string>) : CNF =
    set [ for p in S do
            for q in S do
                if p <> q then
                    set [ Neg p; Neg q ] ]

The function `atLeastOne` takes one argument:
- `S` is a set of propositional variables.

It returns a set of clauses expressing the fact that at least one of the variables of $S$ is `true`.

In [ ]:
let atLeastOne (S: Set<string>) : CNF =
    set [ Set.map Pos S ]

The function `exactlyOne` takes one argument:
- `S` is a set of propositional variables.

It returns a set of clauses expressing the fact that exactly one of the variables in the set $S$ is true.

In [ ]:
let exactlyOne (S: Set<string>) : CNF =
    atMostOne S + atLeastOne S

In [ ]:
exactlyOne (set [ "a"; "b"; "c" ])

The function `bijective` takes three arguments:
- `A` is a set of names,
- `B` is a set of names that has the same number of elements as `A`,
- `f` is the name of a relation.

It returns a set of clauses that is equivalent to the formula
$$   \bigl(\forall x \in A: \exists! y \in B: f\langle x, y\rangle\bigr) \wedge
     \bigl(\forall y \in B: \exists! x \in A: f\langle x, y\rangle\bigr)
$$
Here the expression $f\langle x,y\rangle$ is the name of a propositional variable and the expression $\exists!x:p(x)$ is to be read as "There exists exactly one $x$ such that $p(x)$ holds".

In [ ]:
let bijective (A: Set<string>) (B: Set<string>) (f: string) : CNF =
    failwith "your code here"

In [ ]:
printfn "%s" (cnfToString (bijective (set [ "a"; "b"; "c"; "d" ]) (set [ "x"; "y"; "z"; "u" ]) "f"))

For example, the function call `bijective (set ["a"; "b"; "c"; "d"]) (set ["x"; "y"; "z"; "u"]) "f"` returns a set of 56 clauses: 8 clauses of length 4 and 48 clauses of length 2.

The function `parseAndNormalize` takes one argument:
- `s` is a string representing a propositional formula.

It parses this string as a propositional formula and then turns this formula into a set of clauses.

In [ ]:
let parseAndNormalize (s: string) : CNF =
    normalize (parse s)

The function `exclusiveOr` takes two arguments:
- `a` is a string representing a formula,
- `b` is a string representing a formula.

It computes the *exclusive or* of the formulas $a$ and $b$.  The resulting formula is converted into CNF.

In [ ]:
let exclusiveOr (a: string) (b: string) : CNF =
    let formula = sprintf "((%s) ↔ ¬(%s))" a b
    parseAndNormalize formula

In [ ]:
exclusiveOr "p" "q"

The function `consistentNames` takes three arguments:
- `FirstMale` is a set of male first names,
- `FirstFemale` is a set of female first names,
- `SurNames` is a set of surnames.

It returns a set of clauses that ensures that if the man `x` has the surname `z` and the woman `y` also has the surname `z`, then `x` and `y` have to be married.

In [ ]:
let consistentNames (FirstMale: Set<string>) (FirstFemale: Set<string>) (SurNames: Set<string>) : CNF =
    Set.unionMany [ for x in FirstMale do
                      for y in FirstFemale do
                        for z in SurNames ->
                          parseAndNormalize (sprintf "MaleName<%s,%s> ∧ FemaleName<%s,%s> → Married<%s,%s>" x z y z x y) ]

In [ ]:
(consistentNames FirstMale FirstFemale SurNames).Count

Below, you might need the following symbols: ¬, ∧, ∨, →, ↔

The function `computeClauses` takes three arguments:
- `FirstMale` is a set of male first names,
- `FirstFemale` is a set of female first names,
- `SurNames` is a set of surnames.

It returns the set of clauses describing the puzzle.

In [ ]:
let computeClauses (FirstMale: Set<string>) (FirstFemale: Set<string>) (SurNames: Set<string>) : CNF =
    // Jedem männlichen Vornamen ist genau ein Nachname zugeordnet und umgekehrt.
    let mutable Clauses : CNF = failwith "your code here"
    // Jeder Mann ist mit genau einer Frau verheiratet und umgekehrt.
    Clauses <- Clauses + failwith "your code here"
    // Jede Frau hat genau einen Nachnamen
    Clauses <- Clauses + failwith "your code here"
    // Die Namen sind konsistent.
    Clauses <- Clauses + failwith "your code here"
    // Entweder ist Amanns Vorname Heiner, oder Bemanns Frau heisst Inge.
    Clauses <- Clauses + failwith "your code here"
    // Wenn Cemann mit Josefa verheiratet ist, dann – und nur in diesem Falle –
    // heisst Klaras Mann nicht Fritz.
    Clauses <- Clauses + failwith "your code here"
    // Wenn Josefas Mann nicht Erich heisst, dann ist Inge mit Fritz verheiratet.
    Clauses <- Clauses + failwith "your code here"
    // Wenn Luises Mann Fritz heisst, dann ist der Vorname von Klaras Mann nicht Gustav.
    Clauses <- Clauses + failwith "your code here"
    // Wenn die Frau von Fritz Inge heisst, dann ist Erich nicht mit Josefa verheiratet.
    Clauses <- Clauses + failwith "your code here"
    // Wenn Fritz nicht mit Luise verheiratet ist, dann heisst Gustavs Frau Klara.
    Clauses <- Clauses + failwith "your code here"
    // Entweder ist Demann mit Luise verheiratet, oder Cemann heisst Gustav.
    Clauses <- Clauses + failwith "your code here"
    Clauses

In [ ]:
let Clauses = computeClauses FirstMale FirstFemale SurNames
printfn "%s" (cnfToString Clauses)

There are 242 different clauses.

In [ ]:
Clauses.Count

The function `computeSolution` takes the same three arguments as the function `computeClauses`.  It computes the clauses describing the puzzle and solves them.  The result is a set of unit clauses.

In [ ]:
let computeSolution (FirstMale: Set<string>) (FirstFemale: Set<string>) (SurNames: Set<string>) : CNF =
    let Clauses = computeClauses FirstMale FirstFemale SurNames
    solve Clauses

In [ ]:
#!time
let Solution = computeSolution FirstMale FirstFemale SurNames

The function `onlyPositive` takes one argument:
- `Solution` is a set of unit clauses.

It returns the set of all variables that are true according to `Solution`.  The function `arb` is defined in the notebook `07-Davis-Putnam-JW.ipynb`.

In [ ]:
let onlyPositive (Solution: CNF) : Set<string> =
    set [ for Clause in Solution do
            match arb Clause with
            | Pos v -> v
            | Neg _ -> () ]

In [ ]:
onlyPositive Solution

The functions `extractFirst` and `extractSecond` take one argument:
- `s` is a variable of the form `f<x,y>`.

The function `extractFirst` returns `x`, while `extractSecond` returns `y`.  Both functions use regular expressions.  `Regex.Match` returns the first match of the regular expression in `s`, and `m.Groups[1].Value` is the substring matched by the first parenthesized group of the regular expression.

In [ ]:
open System.Text.RegularExpressions

let extractFirst (s: string) : string =
    let m = Regex.Match(s, "<([A-Za-z]+),")
    m.Groups[1].Value

let extractSecond (s: string) : string =
    let m = Regex.Match(s, ",([A-Za-z]+)>")
    m.Groups[1].Value

The function `displaySolution` takes one argument:
- `Solution` is a set of unit clauses representing a solution of the puzzle.

It prints the solution.  The map `Married` maps every male first name to the first name of his wife, the map `Names` maps every male first name to the corresponding surname.

In [ ]:
let displaySolution (Solution: CNF) : unit =
    let mutable Married : Map<string, string> = Map.empty
    let mutable Names   : Map<string, string> = Map.empty
    for v in onlyPositive Solution do
        if v.StartsWith "Married" then
            Married <- Married.Add(extractFirst v, extractSecond v)
        elif v.StartsWith "MaleName" then
            Names <- Names.Add(extractFirst v, extractSecond v)
    for KeyValue(x, y) in Married do
        printfn "%s %s is married to %s." x Names[x] y

In [ ]:
displaySolution Solution

## Checking the Uniqueness of the Solution

The function `negateSolution` takes one argument:
- `UnitClauses` is a set of unit clauses.

It returns a clause that is the negation of the conjunction of these unit clauses.

In [ ]:
let negateSolution (UnitClauses: CNF) : Clause =
    Set.map (fun Unit -> complement (arb Unit)) UnitClauses

In [ ]:
negateSolution (set [ set [ Pos "a" ]; set [ Neg "b" ] ])

The function `checkUniqueness` takes two arguments:
- `Solution` is a set of unit clauses representing a solution of `Clauses`,
- `Clauses` is a set of clauses.

It checks whether `Solution` is the only solution of `Clauses`.

In [ ]:
let checkUniqueness (Solution: CNF) (Clauses: CNF) : unit =
    let negation    = negateSolution Solution
    let alternative = solve (Clauses.Add negation)
    if alternative = set [ Set.empty ] then
        printfn "Well done: The solution is unique!"
    else
        printfn "ERROR: The solution is not unique!"

In [ ]:
#!time
checkUniqueness Solution Clauses